# Planner prompt iteration

Scratchpad for iterating on the planner's system prompt against the current `inbox/`.
When you find a prompt you like, paste it into `planner.py` and run it from the shell.

Imports live in cell 1; rerun it after editing `planner.py` and autoreload picks up changes.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json
from pathlib import Path

# notebooks/ sits one level below project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
from anthropic import Anthropic

from planner import scan_inbox, build_manifest, SYSTEM_PROMPT, MODEL, MAX_TOKENS

load_dotenv()
client = Anthropic()
print("root:", ROOT)

## 1. What the planner sees

In [ ]:
urls, md_snippets = scan_inbox()
manifest = build_manifest(urls, md_snippets)
print(manifest)

## 2. Run the prompt from `planner.py`

In [ ]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user",
               "content": f"Inbox manifest:\n\n{manifest}\n\n"
                          "Design the taxonomy and the ordered task list."}],
)
text = next(b.text for b in resp.content if b.type == "text")
print(text)
print()
print(f"in={resp.usage.input_tokens} out={resp.usage.output_tokens}")

## 3. Try a variant prompt without touching `planner.py`

Use this cell to A/B alternate prompts against the same inbox. When one wins,
paste it into `planner.py`'s `SYSTEM_PROMPT` and rerun cell 1 (autoreload picks it up).

In [ ]:
VARIANT_PROMPT = SYSTEM_PROMPT  # edit me

resp2 = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=VARIANT_PROMPT,
    messages=[{"role": "user",
               "content": f"Inbox manifest:\n\n{manifest}\n\n"
                          "Design the taxonomy and the ordered task list."}],
)
print(next(b.text for b in resp2.content if b.type == "text"))

## 4. Try parsing the JSON

Phase 2 will need to actually extract `taxonomy` and `tasks` from the response.
Sanity-check that the current prompt produces parseable JSON.

In [ ]:
try:
    plan = json.loads(text)
    print("taxonomy:", plan["taxonomy"])
    print("tasks:")
    for t in plan["tasks"]:
        print(" ", t)
except (json.JSONDecodeError, KeyError) as e:
    print("parse failed:", e)
    print("raw response above")